# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Explore available record sets and fields
print("Available record sets (by @id):")
for rs in dataset.record_sets:
    print(f"  RecordSet name: {rs.name} | @id: {rs.id}")
    print("    Fields (by @id):")
    for field in rs.fields:
        print(f"      Field: {field.name} | @id: {field.id} | DataType: {field.data_type}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Determine record set @ids
record_set_ids = [rs.id for rs in dataset.record_sets]
print(f"RecordSet @ids: {record_set_ids}\n")

dataframes = {}
# Load all record sets as DataFrames
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for RecordSet @id '{record_set_id}' with shape {df.shape}")
    print(f"Columns available (by field @id): {df.columns.tolist()}")
    print()
# Pick the main data table for further analysis (choose the first record set by default)
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"Preview of first 5 rows in main record set ({main_record_set_id}):")
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets found in this dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example EDA: Filter and normalize a numeric field, group by another field

main_df = dataframes[main_record_set_id].copy() if main_record_set_id in dataframes else pd.DataFrame()

if not main_df.empty:
    # Show all numeric fields (@ids)
    print("Numeric fields available (by field @id):")
    numeric_fields = [col for col in main_df.columns if pd.api.types.is_numeric_dtype(main_df[col])]
    print(numeric_fields)
    
    # Default: Pick first numeric field
    numeric_field_id = numeric_fields[0] if numeric_fields else None
    
    if numeric_field_id is not None:
        # Set a threshold (e.g. mean)
        threshold = main_df[numeric_field_id].mean() if main_df[numeric_field_id].dtype != object else 0
        filtered_df = main_df[main_df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (show top 5 rows):")
        display(filtered_df.head())

        # Normalize this numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records (show top 5 rows):")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by a categorical field (pick the first non-numeric field)
        group_fields = [col for col in main_df.columns if col not in numeric_fields]
        group_field_id = group_fields[0] if group_fields else None
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
            display(grouped_df.head())
    else:
        print("No numeric field available for EDA.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the main numeric field
if not main_df.empty and numeric_field_id is not None:
    plt.figure(figsize=(8,5))
    sns.histplot(main_df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.show()

    # Boxplot by group field if possible
    if group_field_id is not None:
        plt.figure(figsize=(12,5))
        sns.boxplot(x=main_df[group_field_id], y=main_df[numeric_field_id])
        plt.title(f"'{numeric_field_id}' by '{group_field_id}'")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No suitable data for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² dataset was loaded using its Croissant schema via the `mlcroissant` library.
- Reviewed the available record sets and fields by their `@id`, ensuring reproducibility and precise reference.
- Performed filtering and normalization on a main numeric field, and grouped data by a relevant attribute for exploratory analysis.
- Visualized distributions to understand data characteristics. For more advanced use cases (e.g., feature engineering, ML, etc.), continue building on the extracted DataFrames and leverage the dataset schema information via `mlcroissant`.

Learn more: [mlcroissant documentation](https://mlcommons.github.io/croissant/)